In [0]:
%run "../libs/libs"

In [0]:
%run "../common/define_path"

In [0]:
otracking_path

In [0]:
def read_bronze_order():
    print('Reading the Bronze Table Data : ',end='')
    df_bronzeorder = (spark.readStream
                    .table(f"otracking.bronze.order")
                    )
    print(f'Reading otracking.bronze.order Success!')
    return df_bronzeorder

In [0]:
df_order = read_bronze_order()

In [0]:
from pyspark.sql.functions import col, to_date
df_order = df_order.select(col("ord").alias("orderno"),
                               col("odate").alias("orderdate"),
                               col("wt").alias("orderweight"),
                               col("ordervalue"),
                               col("productcategory"),
                               col("productname"),
                               col("productsize"),
                               col("productvalue"),
                               col("productweight"),
                               col("noofitems"),
                               col("sendername"),
                               col("sendercity"),
                               col("senderstate"),
                               col("receivername"),
                               col("receivercity"),
                               col("receiverstate"),
                               col("estcost").alias("estimatedcost"),
                               col("shipno").alias("shipmentno"),
                               col("dcost").alias("deliverycost"),
                               col("carriername"),
                               col("carriercity"),
                               col("carrierstate"))

                               

In [0]:
from pyspark.sql.functions import current_timestamp
df_order = df_order.withColumn("ingestion_time",current_timestamp())
df_order = convert_to_timestamp(df_order, "orderdate")

In [0]:
target_columns = ["orderweight", "ordervalue","productvalue","productweight","estimatedcost","deliverycost"]

# Apply the UDF in a loop
for col_name in target_columns:
     df_order = convert_to_float(df_order, col_name)

In [0]:
target_columns = ["productsize", "noofitems"]

# Apply the UDF in a loop
for col_name in target_columns:
     df_order = convert_to_int(df_order, col_name)

In [0]:
def write_silver_order(StreamingDF):
      write_StreamSilver = (StreamingDF.writeStream
                .format('delta')
                .option('checkpointLocation',f'{otracking_path}/silver/checkpoint/order/')
                .outputMode('append')
                .queryName("silverorderstream")
                .trigger(availableNow=True)
                .toTable("otracking.silver.order")
            )
      write_StreamSilver.awaitTermination()

In [0]:
write_silver_order(df_order)

In [0]:
%sql
select * from otracking.silver.order limit 1000